## This notebook demonstrates access to NOAA Coastal Ocean ReAnalysis (CORA) data that is stored in an S3 bucket and made freely available via NOAA's Open Data Dissemination (NODD) service. The data will be loaded as an xarray dataset. For a use case example, water level data is extracted at a bounding box near Myrtle Beach during Hurricane Hugo in 1989. The data is scatter plotted over a basemap tile.

Access all the data in the S3 bucket here: https://noaa-nos-cora-pds.s3.amazonaws.com/index.html

*Parts of the code in this notebook were guided by **Signell, R. (2024). Reproducible Notebooks/hurricane-ike-water-levels. https://github.com/reproducible-notebooks/hurricane-ike-water-levels/tree/master***

_ _

### Import packages ###

In [ ]:
import cartopy.crs as ccrs
import dask
import geoviews as gv
import geoviews.feature as gf
import holoviews as hv
import intake
import numpy as np
import pandas as pd
import xarray as xr
import zarr

hv.extension('matplotlib')

### Initialize and list the available CORA datasets ###
Description of datasets: <br>
**CORA-{domain}-fort.63-{chunking}**: Hourly water levels <br>
**CORA-{domain}-swan_TPS.63-{chunking}**: Hourly spectral peak wave periods <br>
**CORA-{domain}-swan_DIR.63-{chunking}**: Hourly mean wave directions <br>
**CORA-{domain}-swan_HS.63-{chunking}**: Hourly significant wave heights <br>
**CORA-{domain}-Grid-{chunking}**: Hourly water levels interpolated from model nodes to a uniform 500 meter resolution grid <br>

Where: <br>
**{domain}** represents the domain. Options are **GEC** or **PAC**. <br>
**{chunking}** represents the chunking strategy, which affects data loading speed. Use **timeseries** if you are interested in long timeseries at a few points, use **map** if you are interested in many points for a few time periods (up to a few days).

In [ ]:
if int(zarr.__version__[0]) > 2:  ## There are some issues with zarr 2 vs. zarr 3, so two catalogs have been created depdning on the zarr version you are using.
    catalog_file = 'CORA_intake_zarr3.yml'
else:
    catalog_file = 'CORA_intake.yml'

catalog = intake.open_catalog("s3://noaa-nos-cora-pds/"+catalog_file,storage_options={'anon':True})
list(catalog)

### Initialize CORA-GEC water levels (see full dataset [here](https://noaa-nos-cora-pds.s3.amazonaws.com/index.html#cora_gec/native_grid/water_levels/)). ###
#### This dataset contains hourly water levels from 1979-2022 in the GEC domain. The dataset is multiple TB in size and cannot be loaded into memory on a standard computer. We use the xarray package to "lazily" load the data - initializing it and only loading the pieces that we need later. Note that we use the **map** chunking here, since we are interested in a small time slice at many points.

In [ ]:
ds = catalog["CORA-GEC-fort.63-map"].to_dask()
ds

### Select data at model nodes within a bounding box and for a time slice encompassing Hurricane Hugo ###
#### Note that this does not load any data into memory yet! ####

In [ ]:
minX, maxX, minY, maxY = -80.5, -78, 32.5, 34
start_t = "1989-09-22 00:00:00"
end_t = "1989-09-22 23:00:00"

x_subset =  ((ds.x >= minX) & (ds.x <= maxX)).compute()
y_subset = ((ds.y >= minY) & (ds.y <= maxY)).compute() 
boolmask  = (x_subset & y_subset).compute()
nodeIndex = np.where(boolmask)[0] 
zeta_subset = ds['zeta'].sel(time=slice(start_t, end_t)).isel(node=nodeIndex)

### Load the selected data into memory and create a water level map at a specific timetamp ###

In [ ]:
lons = zeta_subset['x'].values
lats = zeta_subset['y'].values
data = zeta_subset.isel(time=5).values
data[np.isnan(data)] = -999
df = pd.DataFrame({'lon':lons,'lat':lats,'zeta':data})

nodes = gv.Points(
    df,
    kdims=['lon', 'lat'],
    vdims=['zeta'],
    crs=ccrs.PlateCarree() # The projection of your input data (Standard Lat/Lon)
).opts(
    c='zeta',
    cmap='turbo', 
    colorbar=True, 
    s=25, 
    marker='o',
    fig_size=250,
    projection=ccrs.Mercator(),
    clim=(1500,3600),
    clabel='mm')

map_fig = gf.ocean * gf.land.opts(facecolor='darkgray') * gf.coastline * gf.states(linewidth=0.5) * nodes
map_fig